# Medical Diagnosis AI — Symptom Parser

## Purpose

This notebook performs the second preprocessing stage: conservative extraction of source-grounded symptom phrases from the cleaned NHS condition dataset.

### Important guarantees

- no invented symptoms or synonyms
- symptom-list headings are never stored as symptom phrases
- incomplete lead-ins such as `Children with sinusitis may:` are treated as headers, not model input
- non-symptom lists such as triggers, treatments, differentials, and comorbidities are stopped where detectable
- compact NHS lists flattened onto one line are recovered conservatively
- uncertain outputs remain marked with `parser_needs_review=True`


# Step 1 — Imports

In [51]:
import json
import re
from pathlib import Path

print("Imports loaded successfully.")

Imports loaded successfully.


# Step 2 — Locate the text-cleaner output
## The normal project location is:
### app/dataset/processed/conditions_text_cleaned.json
## A fallback path is included so this delivered notebook can show preview outputs before you copy it into the project.

In [52]:
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in [cwd, *cwd.parents]
        if (path / "app" / "dataset").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the project root.")

DATASET_PATH = PROJECT_ROOT / "app" / "dataset" / "output" / "preprocessing" / "conditions_text_cleaned.json"

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        "Could not find conditions_text_cleaned.json. "
        "Run 1_text_cleaner.ipynb first."
    )

print("Using cleaned dataset:")
print(DATASET_PATH)

Using cleaned dataset:
C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_text_cleaned.json


# Step 3 — Load and validate the cleaned records

In [53]:
with DATASET_PATH.open("r", encoding="utf-8") as f:
    conditions = json.load(f)

REQUIRED_FIELDS = {
    "condition",
    "symptoms",
    "causes",
    "warnings",
    "recommendations",
    "symptoms_model",
}

assert isinstance(conditions, list), "Dataset must be a JSON list."
assert len(conditions) > 0, "Dataset is empty."

bad_records = [
    record
    for record in conditions
    if not REQUIRED_FIELDS.issubset(record.keys())
]

assert not bad_records, (
    f"{len(bad_records)} record(s) are missing required fields."
)

print(f"Loaded cleaned records: {len(conditions)}")
print("Input validation: PASSED")
print("Required fields:", sorted(REQUIRED_FIELDS))

Loaded cleaned records: 406
Input validation: PASSED
Required fields: ['causes', 'condition', 'recommendations', 'symptoms', 'symptoms_model', 'warnings']


# Step 4 — Select 5 different test conditions
## These conditions give us different symptom-writing styles:
### 1. Acute pancreatitis — narrative + list
### 2. Acne — descriptive list items
### 3. Scoliosis — signs + short narrative
### 4. Aplastic anaemia — multiple symptom sections
### 5. Acute respiratory infection (ARI) — common + less common symptom lists

In [54]:
DEMO_CONDITIONS = [
    "Acute pancreatitis",
    "Acne",
    "Scoliosis",
    "Aplastic anaemia",
    "Acute respiratory infection (ARI)",
]

demo_records = []

for name in DEMO_CONDITIONS:
    record = next(
        (r for r in conditions if r["condition"] == name),
        None,
    )

    if record is None:
        raise ValueError(f"Demo condition not found: {name}")

    demo_records.append(record)

print("Demo conditions loaded:")
for record in demo_records:
    print("-", record["condition"])

Demo conditions loaded:
- Acute pancreatitis
- Acne
- Scoliosis
- Aplastic anaemia
- Acute respiratory infection (ARI)


# Step 5 — Phrase normalization
## This function performs only light formatting cleanup on extracted phrases.
## It does not invent synonyms, stem words, or change the medical meaning.

In [55]:
def normalize_phrase(phrase: str) -> str:
    if not phrase:
        return ""

    phrase = phrase.strip()

    # Collapse repeated whitespace.
    phrase = re.sub(r"\s+", " ", phrase)

    # Remove spaces before punctuation.
    phrase = re.sub(r"\s+([,.;:])", r"\1", phrase)

    # Remove surrounding list/formatting punctuation.
    phrase = phrase.strip(" \t\n-–—:;,.")

    return phrase


demo_phrase = "   -  severe   abdominal pain   "
print("Before:", repr(demo_phrase))
print("After: ", repr(normalize_phrase(demo_phrase)))

Before: '   -  severe   abdominal pain   '
After:  'severe abdominal pain'


# Step 6 — Detect likely list-style symptom items
## The NHS symptom text often contains useful line-separated items.
## We keep short symptom-like lines and reject obvious:
### - headers ending in :
### - explanatory/context sentences
### - navigation/guidance text
### - long narrative sentences
## This is deliberately conservative.

In [56]:

# -----------------------------
# Symptom section / line rules
# -----------------------------

SYMPTOM_HEADER_RE = re.compile(
    r"^(?:"
    r"(?:common |less common |other |initial |further |early |later |main |most common )?"
    r"symptoms?(?: of [^:]+?)?"
    r"(?: can| may| might| could)? "
    r"(?:include|including|like|are|is)"
    r"|"
    r"(?:common |less common |other |initial |further |early |later |main |most common )?"
    r"signs?(?: of [^:]+?)?"
    r"(?: can| may| might| could)? "
    r"(?:include|including|like|are|is)"
    r"|"
    r"typical signs include"
    r"|"
    r"people may experience"
    r"|"
    r"patients may experience"
    r"|"
    r"you may experience"
    r"|"
    r"you may go on to develop"
    r"|"
    r"some patients.*symptoms.*like"
    r")",
    flags=re.IGNORECASE,
)

# These headings introduce causes, triggers, differentials,
# treatments, comorbidities, or advice rather than symptoms.
NON_SYMPTOM_LIST_HEADER_RE = re.compile(
    r"(?:"
    r"mistaken for|"
    r"occur(?:s)? alongside|"
    r"alongside other (?:problems|conditions)|"
    r"other (?:problems|health )?conditions|"
    r"health conditions like|"
    r"triggered by|can be triggered|"
    r"triggers?(?: include| are)|"
    r"things can make .+ worse|"
    r"make the .+ worse,\s*including|"
    r"risk factors?|"
    r"treatments?(?: include| are| may)|"
    r"medications?(?: include| are)|"
    r"ways to (?:treat|manage|prevent)|"
    r"things you can do|what you can do|"
    r"complications?(?: include| are| can)|"
    r"causes? of [^:]+(?: include| are)|"
    r"possible causes?|caused by|"
    r"different types of treatment|"
    r"everyday tasks,\s*such as"
    r")",
    flags=re.IGNORECASE,
)

GENERIC_LEADIN_RE = re.compile(
    r"^(?:"
    r"these|those|this|it|for example|examples?|"
    r"the following|these problems|these conditions"
    r")\s+(?:can|may|might|could|will|would|do|are|"
    r"include|includes|including|happen|occur|lead|cause|make|mean)\b",
    flags=re.IGNORECASE,
)

ACTION_START_RE = re.compile(
    r"^(?:"
    r"wash|use|take|avoid|stop|eat|exercise|try|"
    r"contact|phone|call|stay|wear|tell|let|read|"
    r"find|rest|drink|apply|keep|seek|speak|"
    r"if you\'re worried|if you are worried|"
    r"if you\'re concerned|if you are concerned|"
    r"do not|don't|you should|you must"
    r")\b",
    flags=re.IGNORECASE,
)

# While we are already inside a confirmed symptom list, use a
# narrower advice detector. Symptom items can legitimately start
# with verbs such as "find", "avoid", or "feel".
ACTIVE_ADVICE_RE = re.compile(
    r"^(?:"
    r"contact|phone|call|seek|speak to|tell your|"
    r"you should|you must|you need to|"
    r"do not|don't|"
    r"treatment|treatments|self[- ]care|"
    r"read more|find out more"
    r")\b",
    flags=re.IGNORECASE,
)

STOP_SECTION_RE = re.compile(
    r"^(?:"
    r"there are things you can do|"
    r"things you can do|"
    r"a number of treatments|"
    r"treatment|treatments|"
    r"self[- ]care|"
    r"how to|"
    r"to reduce|"
    r"to prevent|"
    r"prevention|"
    r"depending on the cause|"
    r"exercising too often|"
    r"advice|"
    r"what you can do|"
    r"making changes to"
    r")\b",
    flags=re.IGNORECASE,
)

CONTEXT_START_RE = re.compile(
    r"^(?:"
    r"these symptoms|"
    r"those symptoms|"
    r"this symptom|"
    r"everyone|"
    r"people can have|"
    r"there is|there are|"
    r"it's important|it is important|"
    r"making changes|"
    r"depending on"
    r")\b",
    flags=re.IGNORECASE,
)

META_PHRASE_RE = re.compile(
    r"^(?:"
    r"one or multiple symptoms|"
    r"your symptoms(?: to)?|"
    r"symptoms may vary|"
    r"symptoms can vary|"
    r"different symptoms|"
    r"multiple symptoms|"
    r"typical signs include|"
    r"people may experience|"
    r"patients may experience|"
    r"you may experience|"
    r"you may go on to develop|"
    r"(?:common |less common |other |initial |further |early |later )?"
    r"symptoms?(?: of [^:]+?)?(?: can| may| might| could)? "
    r"(?:include|including|like|are|is)"
    r")$",
    flags=re.IGNORECASE,
)

INCOMPLETE_LEADIN_RE = re.compile(
    r"^(?:"
    r"(?:these|those|they|this|it|for example)\b.*"
    r"|"
    r".*\b(?:may|might|can|could|will|would)\s*$"
    r"|"
    r".*\b(?:include|includes|including|such as|for example|"
    r"experience|have|develop|notice|feel)\s*$"
    r")",
    flags=re.IGNORECASE,
)


def is_non_symptom_list_header(line: str) -> bool:
    if not line:
        return False

    return bool(
        NON_SYMPTOM_LIST_HEADER_RE.search(
            line.strip().rstrip(":")
        )
    )


def is_symptom_header(line: str) -> bool:
    if not line:
        return False

    raw = line.strip()
    text = raw.rstrip(":").strip()

    if is_non_symptom_list_header(text):
        return False

    if re.search(
        r"\b(?:mistaken for|no symptoms?|without symptoms?)\b",
        text,
        flags=re.IGNORECASE,
    ):
        return False

    if SYMPTOM_HEADER_RE.search(text):
        return True

    if re.search(
        r"^the (?:main|most common) (?:symptoms?|signs?) "
        r"of .+ (?:include|including|are|is)$",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    if re.search(
        r".+\b(?:symptoms?|signs?)\s+"
        r"(?:can|may|might|could)?\s*"
        r"(?:include|including|are|is|like)$",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    # The broader rules below are only trusted for a source
    # line that explicitly introduces a list with a colon.
    if not raw.endswith(":"):
        return False

    # "These can include:" is too ambiguous to restart a
    # symptom section on its own.
    if GENERIC_LEADIN_RE.match(text):
        return False

    if re.search(
        r"\b(?:symptoms?|signs?|tics?|sensations?)\b",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    # NHS often uses headers such as:
    # "Children with sinusitis may:"
    # "If you have insomnia, you might:"
    if re.search(
        r"\b(?:may|might|can|could|will|would)\s*$",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    if (
        re.search(
            r"\b(?:may|might|can|could|will|would)\b",
            text,
            flags=re.IGNORECASE,
        )
        and re.search(
            r"\b(?:have|experience|develop|notice|feel|find|"
            r"be|become|cause|lead to|mean|affect|hear|see)\b",
            text,
            flags=re.IGNORECASE,
        )
    ):
        return True

    if re.search(
        r"\b(?:likely to).*(?:experience|have|develop|notice|feel)$",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    # Presentation-style headings such as:
    # "Cellulitis causes an area of skin to suddenly become:"
    # "Catarrh can lead to a:"
    if re.search(
        r"\b(?:causes?|leads? to|makes?)\b",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    if re.search(
        r"\b(?:tend to be|likely to be)\s*$",
        text,
        flags=re.IGNORECASE,
    ):
        return True

    return False


def is_action_or_advice(line: str) -> bool:
    if not line:
        return False

    return bool(
        ACTION_START_RE.match(
            line.strip()
        )
    )


def is_active_advice(line: str) -> bool:
    if not line:
        return False

    return bool(
        ACTIVE_ADVICE_RE.match(
            line.strip()
        )
    )


def is_stop_section(line: str) -> bool:
    if not line:
        return False

    return bool(
        STOP_SECTION_RE.match(
            line.strip()
        )
    )


def looks_like_prose_boundary(line: str) -> bool:
    if not line:
        return False

    line = line.strip()

    if re.search(
        r"[.!?]\s+\S",
        line,
    ):
        return True

    if (
        line[-1:] in ".!?"
        and len(line.split()) > 10
    ):
        return True

    return False


def is_incomplete_leadin_phrase(phrase: str) -> bool:
    phrase = normalize_phrase(phrase)

    if not phrase:
        return True

    if re.search(
        r"\b(?:no|without)\s+(?:noticeable\s+)?symptoms?\b",
        phrase,
        flags=re.IGNORECASE,
    ):
        return True

    return bool(
        INCOMPLETE_LEADIN_RE.match(
            phrase
        )
    )


def is_probable_list_item(line: str) -> bool:
    if not line:
        return False

    raw = line.strip()

    if not raw:
        return False

    if raw.endswith(":"):
        return False

    if is_symptom_header(raw):
        return False

    if is_action_or_advice(raw):
        return False

    if is_stop_section(raw):
        return False

    if CONTEXT_START_RE.match(raw):
        return False

    phrase = normalize_phrase(raw)

    if not phrase:
        return False

    if META_PHRASE_RE.match(phrase):
        return False

    if is_incomplete_leadin_phrase(phrase):
        return False

    if len(phrase) > 220:
        return False

    if (
        raw[-1:] in ".!?"
        and len(phrase.split()) > 10
    ):
        return False

    return True


# Step 7 — Extract high-confidence symptom phrases from narrative sentences
## Some conditions have an important symptom written inside prose rather than a bullet list.
## For now we only use a few high-confidence patterns, such as:
### - The main symptom ... is ...
### - People with ... may also have ...
## We intentionally do not try to understand every sentence yet.

In [57]:

def split_sentences(text: str) -> list[str]:
    """
    Split prose into sentences WITHOUT destroying line boundaries.

    NHS pages often use newlines to represent symptom lists,
    so those boundaries must be preserved.
    """

    if not text:
        return []

    sentences = []

    for raw_line in text.splitlines():

        line = re.sub(
            r"\s+",
            " ",
            raw_line,
        ).strip()

        if not line:
            continue

        parts = re.split(
            r"(?<=[.!?])\s+",
            line,
        )

        for part in parts:
            part = part.strip()

            if part:
                sentences.append(part)

    return sentences


def trailing_segment(line: str) -> str:
    if not line:
        return ""

    parts = re.split(
        r"(?<=[.!?])\s+",
        line.strip(),
    )

    if len(parts) < 2:
        return ""

    return parts[-1].strip()


def has_trailing_symptom_header(line: str) -> bool:
    """
    Detect a symptom-list heading that appears after prose on the
    same cleaned source line.
    """

    tail = trailing_segment(line)

    if not tail:
        return False

    if is_symptom_header(tail):
        return True

    # NHS example:
    # "Golfer's elbow causes pain and tenderness... This can cause:"
    if re.fullmatch(
        r"this can cause\s*:?",
        tail,
        flags=re.IGNORECASE,
    ):
        prefix = line[: line.rfind(tail)].strip()

        return bool(
            re.search(
                r"\b(?:pain|tenderness|symptoms?|signs?|"
                r"swelling|stiffness|rash|difficulty)\b",
                prefix,
                flags=re.IGNORECASE,
            )
        )

    # NHS example:
    # "Symptoms can be vague ... if you notice the following:"
    if re.search(
        r"(?:notice|experience|have)\s+the following\s*:$",
        tail,
        flags=re.IGNORECASE,
    ):
        prefix = line[: line.rfind(tail)].strip()

        return bool(
            re.search(
                r"\b(?:symptoms?|signs?)\b",
                prefix,
                flags=re.IGNORECASE,
            )
        )

    # Generic transitions are accepted only when the preceding
    # sentence explicitly says they are symptoms/signs.
    if re.fullmatch(
        r"(?:these|they|this)\s+(?:can|may|might|could)\s+include\s*:",
        tail,
        flags=re.IGNORECASE,
    ):
        prefix = line[: line.rfind(tail)].strip()

        return bool(
            re.search(
                r"\b(?:symptoms?|signs?)\b",
                prefix,
                flags=re.IGNORECASE,
            )
            and not re.search(
                r"\b(?:other problems|other conditions|comorbid|"
                r"risk factors?|causes?)\b",
                prefix,
                flags=re.IGNORECASE,
            )
        )

    if re.fullmatch(
        r"this\s+(?:could|can|may|might)\s+be\s*:",
        tail,
        flags=re.IGNORECASE,
    ):
        prefix = line[: line.rfind(tail)].strip()

        return bool(
            re.search(
                r"\b(?:symptoms?|signs?)\b",
                prefix,
                flags=re.IGNORECASE,
            )
        )

    # NHS pages sometimes use a very short transition after
    # explanatory prose, for example "Some:" or "Statistically:".
    # Trust these only when the preceding sentence provides a
    # clear presentation context.
    if re.fullmatch(
        r"(?:some|statistically)\s*:",
        tail,
        flags=re.IGNORECASE,
    ):
        prefix = line[: line.rfind(tail)].strip()

        return bool(
            re.search(
                r"\b(?:symptoms?|signs?|health conditions|"
                r"different ways|presentation|features?)\b",
                prefix,
                flags=re.IGNORECASE,
            )
        )

    return False


def has_trailing_non_symptom_header(line: str) -> bool:
    tail = trailing_segment(line)

    if not tail:
        return False

    if not tail.endswith(":"):
        return False

    return not has_trailing_symptom_header(line)


def extract_trailing_fragment(line: str) -> str:
    """
    Recover a first symptom accidentally attached to prose.

    Colon-introduced list headings are NEVER returned as symptom
    phrases. This is the key protection against fragments such as
    "Children with sinusitis may" becoming model input.
    """

    tail = trailing_segment(line)

    if not tail:
        return ""

    if tail.endswith(":"):
        return ""

    if is_symptom_header(tail):
        return ""

    if tail[-1:] in ".!?":
        return ""

    if len(tail.split()) > 30:
        return ""

    if is_action_or_advice(tail):
        return ""

    if is_stop_section(tail):
        return ""

    if CONTEXT_START_RE.match(tail):
        return ""

    phrase = normalize_phrase(tail)

    if META_PHRASE_RE.match(phrase):
        return ""

    if is_incomplete_leadin_phrase(phrase):
        return ""

    return phrase


def extract_narrative_symptoms(text: str) -> list[str]:
    """
    Extract only high-confidence symptoms from prose.

    Broad 'can cause...' extraction is intentionally avoided because
    it can capture complications or treatment effects.
    """

    if not text:
        return []

    extracted = []

    patterns = [

        # "The main symptom ... is ..."
        re.compile(
            r"\b(?:main|most common) symptom"
            r"(?: of [^.!?]+?)? is (.+)",
            re.IGNORECASE,
        ),

        # "Pain and loss of movement are the main symptoms of ..."
        re.compile(
            r"^(.{3,180}?)\s+(?:is|are)\s+"
            r"(?:the\s+)?(?:main|most common)\s+symptoms?\s+of\b",
            re.IGNORECASE,
        ),

        # "Symptoms may/can include ..."
        re.compile(
            r"\bsymptoms?"
            r"(?: of [^.!?]+?)?"
            r" (?:can|may|might) include (.+)",
            re.IGNORECASE,
        ),

        # "Signs may/can include ..."
        re.compile(
            r"\bsigns?"
            r"(?: can| may| might)"
            r" include (.+)",
            re.IGNORECASE,
        ),

        # "You're likely to experience ..."
        re.compile(
            r"\b(?:you're|you are)"
            r" likely to experience (.+)",
            re.IGNORECASE,
        ),

        # "People with scoliosis may also have back pain."
        re.compile(
            r"\b(?:people|patients) with [^.!?]+?"
            r" may also "
            r"(?:have|experience) (.+)",
            re.IGNORECASE,
        ),

        # "Some people may also experience..."
        re.compile(
            r"\bsome people "
            r"(?:may|might) "
            r"(?:also )?"
            r"(?:have|experience|develop) (.+)",
            re.IGNORECASE,
        ),
    ]

    for sentence in split_sentences(text):

        if is_action_or_advice(sentence):
            continue

        if is_stop_section(sentence):
            continue

        if CONTEXT_START_RE.match(sentence):
            continue

        for pattern in patterns:

            match = pattern.search(sentence)

            if not match:
                continue

            phrase = normalize_phrase(
                match.group(1)
            )

            if not phrase:
                break

            if is_active_advice(phrase):
                break

            if is_stop_section(phrase):
                break

            if META_PHRASE_RE.match(phrase):
                break

            if is_incomplete_leadin_phrase(phrase):
                break

            if len(phrase.split()) > 30:
                break

            extracted.append(phrase)

            break

    return extracted


def split_compact_symptom_line(line: str) -> list[str]:
    """
    Recover NHS lists that were flattened onto one source line:

    "joint pain. stiffness. swelling."

    This is only used for the FIRST item after a confirmed symptom
    header, which prevents ordinary explanatory prose from being
    mistaken for a list.
    """

    parts = re.split(
        r"(?<=[.!?])\s+",
        line.strip(),
    )

    if len(parts) <= 1:
        return []

    output = []

    for part in parts:

        phrase = normalize_phrase(part)

        if not phrase:
            continue

        if len(phrase.split()) > 25:
            return []

        if is_active_advice(phrase):
            return []

        if is_incomplete_leadin_phrase(phrase):
            return []

        output.append(phrase)

    if len(output) < 2:
        return []

    return output


def extract_plain_list_fallback(text: str) -> list[str]:
    """
    Fallback for NHS pages where symptoms are already line-separated
    but the page does not use one of the explicit heading patterns.

    It stays conservative about prose, headings and incomplete lead-ins.
    """

    if not text:
        return []

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) < 2:
        return []

    candidates = []
    active = False
    explicit_header_seen = False
    section_item_count = 0

    for line in lines:

        if has_trailing_symptom_header(line):
            active = True
            explicit_header_seen = True
            section_item_count = 0
            continue

        if has_trailing_non_symptom_header(line):
            active = False
            section_item_count = 0
            continue

        if line.endswith(":"):

            if is_non_symptom_list_header(line):
                active = False
                section_item_count = 0
                continue

            if is_symptom_header(line):
                active = True
                explicit_header_seen = True
                section_item_count = 0
                continue

            # Unknown heading itself is never a symptom.
            active = False
            section_item_count = 0
            continue

        if active:

            if section_item_count == 0:
                compact = split_compact_symptom_line(line)

                if compact:
                    candidates.extend(compact)
                    section_item_count += len(compact)
                    continue

            if looks_like_prose_boundary(line):
                active = False
                section_item_count = 0
                continue

            phrase = normalize_phrase(line)

            if (
                phrase
                and not is_active_advice(phrase)
                and not is_stop_section(phrase)
                and not META_PHRASE_RE.match(phrase)
                and not is_incomplete_leadin_phrase(phrase)
                and len(phrase) <= 220
            ):
                candidates.append(phrase)
                section_item_count += 1

            continue

        # Headerless recovery. Do not treat long prose as a list item,
        # but keep looking because some NHS pages put a clean list later.
        if looks_like_prose_boundary(line):
            continue

        if is_probable_list_item(line):
            candidates.append(
                normalize_phrase(line)
            )

    if explicit_header_seen:
        return candidates

    if len(candidates) >= 2:
        return candidates

    return []


narrative_demo = (
    "The main symptom of acute pancreatitis is a severe, dull pain "
    "around the top of your stomach that develops suddenly. "
    "People with scoliosis may also have back pain."
)

print(
    extract_narrative_symptoms(
        narrative_demo
    )
)


['a severe, dull pain around the top of your stomach that develops suddenly', 'back pain']


In [58]:
# ---------------------------------------------------------
# Narrative fallback — FINAL CLEANUP VERSION
# ---------------------------------------------------------

GENERIC_FALLBACK_PHRASES = {
    "symptom",
    "symptoms",
    "a symptom",
    "the symptoms",
    "a range of symptoms",
    "range of symptoms",
    "different symptoms",
    "disease",
    "a disease",
    "condition",
    "a condition",
    "problems",
    "a problem",
    "changes",
    "effects",
}


GENERIC_FALLBACK_RE = re.compile(
    r"^(?:"
    r"(?:a |the )?(?:range of )?symptoms?"
    r"|"
    r"(?:a |the )?disease"
    r"|"
    r"(?:a |the )?condition"
    r"|"
    r"(?:a |the )?problem(?:s)?"
    r"|"
    r"changes"
    r"|"
    r"effects"
    r")$",
    flags=re.IGNORECASE,
)


WEAK_MODIFIER_RE = re.compile(
    r"^(?:"
    r"often worse|"
    r"usually worse|"
    r"worse at|"
    r"better at|"
    r"more noticeable|"
    r"less noticeable|"
    r"more common|"
    r"less common|"
    r"can vary|"
    r"may vary"
    r")\b",
    flags=re.IGNORECASE,
)


def clean_narrative_candidate(phrase: str) -> str:
    """
    Conservative cleanup for symptoms extracted from prose.
    """

    if not phrase:
        return ""

    phrase = normalize_phrase(phrase)

    if not phrase:
        return ""

    # Remove trailing advice after otherwise useful text.
    phrase = re.split(
        r"\b(?:but|however),?\s+"
        r"(?:it's|it is)?\s*important to\b",
        phrase,
        maxsplit=1,
        flags=re.IGNORECASE,
    )[0]

    phrase = normalize_phrase(phrase)

    if not phrase:
        return ""

    # Reject obvious generic/non-medical outputs.
    if phrase.casefold() in GENERIC_FALLBACK_PHRASES:
        return ""

    if GENERIC_FALLBACK_RE.match(phrase):
        return ""

    # Reject modifier-only phrases.
    if WEAK_MODIFIER_RE.match(phrase):
        return ""

    if is_action_or_advice(phrase):
        return ""

    if is_stop_section(phrase):
        return ""

    if META_PHRASE_RE.match(phrase):
        return ""

    # A huge capture probably means the regex failed.
    if len(phrase.split()) > 35:
        return ""

    return phrase


def extract_narrative_fallback(text: str) -> list[str]:
    """
    Runs ONLY when structured parsing and plain-list parsing fail.

    Uses conservative source-grounded prose patterns.
    """

    if not text:
        return []

    candidates = []

    patterns = [

        # -------------------------------------------------
        # "causes symptoms such as pain, numbness..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:cause|causes|causing|caused)\s+"
            r"symptoms?\s+(?:such as|like)\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "symptoms which include bone pain..."
        # "symptoms include..."
        # -------------------------------------------------

        re.compile(
            r"\bsymptoms?\s+"
            r"(?:which\s+)?"
            r"(?:can\s+|may\s+|might\s+)?"
            r"include\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "symptoms such as..."
        # -------------------------------------------------

        re.compile(
            r"\bsymptoms?\s+such as\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "commonly presents as..."
        # "may present with..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:commonly\s+)?"
            r"(?:presents?|presenting)\s+"
            r"(?:as|with)\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "can present as..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:can|may|might)\s+"
            r"present\s+(?:as|with)\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "can result in feelings of weakness..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:can|may|might)\s+"
            r"result\s+in\s+"
            r"(?:feelings?\s+of\s+)?(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "Some women may notice..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:you|people|patients|"
            r"some people|some women|some men|"
            r"women|men)\s+"
            r"(?:may|might|can)\s+"
            r"notice\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "People may experience..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:you|people|patients|"
            r"some people|some women|some men)\s+"
            r"(?:may|might|can)\s+"
            r"experience\s+(.+)",
            re.IGNORECASE,
        ),

        # -------------------------------------------------
        # "Pain may spread/radiate..."
        # -------------------------------------------------

        re.compile(
            r"\b(?:pain|discomfort)\s+"
            r"(?:can|may|might)\s+"
            r"(?:spread|radiate)\s+(.+)",
            re.IGNORECASE,
        ),
    ]

    for sentence in split_sentences(text):

        sentence = sentence.strip()

        if not sentence:
            continue

        if is_action_or_advice(sentence):
            continue

        if is_stop_section(sentence):
            continue

        # Do not extract from statements explicitly
        # saying that there are no symptoms.
        if re.search(
            r"\b(?:no|without)\s+"
            r"(?:noticeable\s+)?symptoms?\b",
            sentence,
            flags=re.IGNORECASE,
        ):
            continue

        for pattern in patterns:

            match = pattern.search(sentence)

            if not match:
                continue

            phrase = clean_narrative_candidate(
                match.group(1)
            )

            if phrase:
                candidates.append(phrase)

            break

    # Exact deduplication.
    result = []
    seen = set()

    for phrase in candidates:

        phrase = normalize_phrase(phrase)

        if not phrase:
            continue

        key = phrase.casefold()

        if key not in seen:
            seen.add(key)
            result.append(phrase)

    return result

# Step 8 — Combine list extraction + narrative extraction
## This is the first version of the parser.
## The important rule is: prefer missing a difficult phrase over inventing or misclassifying medical information.
## Any condition the parser cannot handle confidently can later be flagged for review.

In [59]:

def parse_symptoms(
    text: str,
    return_method: bool = False,
):
    if not text:
        if return_method:
            return [], "none"

        return []

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    candidates = []
    parser_method = "structured"

    # ----------------------------------
    # 1. High-confidence narrative
    # ----------------------------------

    candidates.extend(
        extract_narrative_symptoms(text)
    )

    # ----------------------------------
    # 2. Structured symptom lists
    # ----------------------------------

    in_symptom_section = False
    section_item_count = 0

    for line in lines:

        # A symptom-list heading can occur after prose on the
        # same cleaned source line.
        if has_trailing_symptom_header(line):
            in_symptom_section = True
            section_item_count = 0
            continue

        # A different kind of colon heading ends the current
        # symptom section.
        if has_trailing_non_symptom_header(line):
            in_symptom_section = False
            section_item_count = 0
            continue

        if line.endswith(":"):

            if is_non_symptom_list_header(line):
                in_symptom_section = False
                section_item_count = 0
                continue

            if is_symptom_header(line):
                in_symptom_section = True
                section_item_count = 0
                continue

            # Unknown colon heading: do not treat the heading or
            # the following list as symptoms.
            in_symptom_section = False
            section_item_count = 0
            continue

        if in_symptom_section:

            # Some NHS pages flatten an entire symptom list into
            # one period-separated line. Only recover this on the
            # first item after a confirmed symptom header.
            if section_item_count == 0:

                compact = split_compact_symptom_line(
                    line
                )

                if compact:
                    candidates.extend(compact)
                    section_item_count += len(compact)
                    continue

            # Once at least one list item has been seen, a prose
            # paragraph marks the end of the symptom list.
            if looks_like_prose_boundary(line):
                in_symptom_section = False
                section_item_count = 0
                continue

            phrase = normalize_phrase(line)

            if (
                phrase
                and not is_active_advice(phrase)
                and not is_stop_section(phrase)
                and not META_PHRASE_RE.match(phrase)
                and not is_incomplete_leadin_phrase(phrase)
                and len(phrase) <= 220
            ):
                candidates.append(phrase)
                section_item_count += 1
                continue

            if (
                is_active_advice(line)
                or is_stop_section(line)
            ):
                in_symptom_section = False
                section_item_count = 0

            continue

        # ----------------------------------
        # 3. Recover one attached symptom
        # ----------------------------------

        if (
            is_action_or_advice(line)
            or is_stop_section(line)
        ):
            continue

        trailing = extract_trailing_fragment(
            line
        )

        if trailing:
            candidates.append(trailing)
            in_symptom_section = True
            section_item_count = 1

    # ----------------------------------
    # 4. Headerless list fallback
    # ----------------------------------

    if not candidates:

        plain_list = extract_plain_list_fallback(
            text
        )

        if plain_list:
            candidates.extend(plain_list)
            parser_method = "plain_list_fallback"

    # ----------------------------------
    # 5. Narrative prose fallback
    # ----------------------------------

    if not candidates:

        narrative = extract_narrative_fallback(
            text
        )

        if narrative:
            candidates.extend(narrative)
            parser_method = "narrative_fallback"

    # ----------------------------------
    # 6. Final filtering + deduplication
    # ----------------------------------

    result = []
    seen = set()

    for phrase in candidates:

        phrase = normalize_phrase(phrase)

        if not phrase:
            continue

        if is_active_advice(phrase):
            continue

        if is_stop_section(phrase):
            continue

        if META_PHRASE_RE.match(phrase):
            continue

        if is_symptom_header(phrase):
            continue

        if is_incomplete_leadin_phrase(phrase):
            continue

        if len(phrase.split()) > 35:
            continue

        key = phrase.casefold()

        if key not in seen:
            seen.add(key)
            result.append(phrase)

    if not result:
        parser_method = "none"

    if return_method:
        return result, parser_method

    return result


test_text = """Other symptoms can include:
nausea
vomiting
vomiting
shortness of breath
"""

print(
    parse_symptoms(
        test_text
    )
)


['nausea', 'vomiting', 'shortness of breath']


# Step 9 — Testing Before/after

## Acute pancreatitis

In [60]:
record = next(r for r in demo_records if r["condition"] == "Acne")

print("BEFORE:\n")
print(record["symptoms"])

print("\nAFTER — symptom phrases:\n")
for phrase in parse_symptoms(record["symptoms"]):
    print("-", phrase)

BEFORE:

Acne can cause different kinds of spots, including:
blackheads - small black or yellowish bumps on the skin
whiteheads - similar to blackheads but may be firmer
papules - small red bumps that may feel tender or sore
pustules - similar to papules, but have a white tip in the centre caused by a build-up of pus
nodules - large hard lumps that build up beneath the surface of the skin and can be painful
cysts - severe acne can cause nodules and cysts which are large pus-filled lumps that can lead to scarring

AFTER — symptom phrases:

- blackheads - small black or yellowish bumps on the skin
- whiteheads - similar to blackheads but may be firmer
- papules - small red bumps that may feel tender or sore
- pustules - similar to papules, but have a white tip in the centre caused by a build-up of pus
- nodules - large hard lumps that build up beneath the surface of the skin and can be painful
- cysts - severe acne can cause nodules and cysts which are large pus-filled lumps that can lea

## Acne

In [61]:
record = next(r for r in demo_records if r["condition"] == "Acne")

print("BEFORE:\n")
print(record["symptoms"])

print("\nAFTER — symptom phrases:\n")
for phrase in parse_symptoms(record["symptoms"]):
    print("-", phrase)

BEFORE:

Acne can cause different kinds of spots, including:
blackheads - small black or yellowish bumps on the skin
whiteheads - similar to blackheads but may be firmer
papules - small red bumps that may feel tender or sore
pustules - similar to papules, but have a white tip in the centre caused by a build-up of pus
nodules - large hard lumps that build up beneath the surface of the skin and can be painful
cysts - severe acne can cause nodules and cysts which are large pus-filled lumps that can lead to scarring

AFTER — symptom phrases:

- blackheads - small black or yellowish bumps on the skin
- whiteheads - similar to blackheads but may be firmer
- papules - small red bumps that may feel tender or sore
- pustules - similar to papules, but have a white tip in the centre caused by a build-up of pus
- nodules - large hard lumps that build up beneath the surface of the skin and can be painful
- cysts - severe acne can cause nodules and cysts which are large pus-filled lumps that can lea

## Scoliosis

In [62]:
record = next(r for r in demo_records if r["condition"] == "Scoliosis")

print("BEFORE:\n")
print(record["symptoms"])

print("\nAFTER — symptom phrases:\n")
for phrase in parse_symptoms(record["symptoms"]):
    print("-", phrase)

BEFORE:

Scoliosis is usually first noticed by a change in the appearance of the back. Typical signs include:
a visibly curved spine
one shoulder being higher than the other
one shoulder or hip being more prominent than the other
clothes not hanging properly
a prominent ribcage
a difference in leg lengths
People with scoliosis may also have back pain. This is more common in adults.

AFTER — symptom phrases:

- back pain
- a visibly curved spine
- one shoulder being higher than the other
- one shoulder or hip being more prominent than the other
- clothes not hanging properly
- a prominent ribcage
- a difference in leg lengths


## Aplastic anaemia

In [63]:
record = next(r for r in demo_records if r["condition"] == "Aplastic anaemia")

print("BEFORE:\n")
print(record["symptoms"])

print("\nAFTER — symptom phrases:\n")
for phrase in parse_symptoms(record["symptoms"]):
    print("-", phrase)

BEFORE:

Symptoms of aplastic anaemia can vary in severity. They are caused by low blood counts, which affect red blood cells, white blood cells and platelets. Symptoms of aplastic anaemia can include:
fatigue
shortness of breath
headaches
chest pains
bleeding
bruising
reoccurring severe infections
Some patients also report experiencing general symptoms like:
brain fog
poor concentration
These symptoms may be caused by aplastic anaemia, different treatments, or other factors.

AFTER — symptom phrases:

- fatigue
- shortness of breath
- headaches
- chest pains
- bleeding
- bruising
- reoccurring severe infections
- brain fog
- poor concentration


## Acute respiratory infection (ARI)

In [64]:
record = next(
    r for r in demo_records
    if r["condition"] == "Acute respiratory infection (ARI)"
)

print("BEFORE:\n")
print(record["symptoms"])

print("\nAFTER — symptom phrases:\n")
for phrase in parse_symptoms(record["symptoms"]):
    print("-", phrase)

BEFORE:

Common symptoms of ARIs include:
a new continuous cough
sneezing
a blocked or runny nose
a sore throat
a headache
muscle aches
shortness of breath , tight chest or wheezing
a high temperature ( fever ) - this is usually about 38°C or higher (100.4°F)
feeling generally unwell
feeling tired or having a lack of energy
Less common symptoms of ARIs include:
diarrhoea
loss of appetite
feeling sick or vomiting
Some respiratory infections, like the common cold and RSV, may also lead to middle ear infection , sinusitis and, in younger children, croup .

AFTER — symptom phrases:

- a new continuous cough
- sneezing
- a blocked or runny nose
- a sore throat
- a headache
- muscle aches
- shortness of breath, tight chest or wheezing
- a high temperature ( fever ) - this is usually about 38°C or higher (100.4°F)
- feeling generally unwell
- feeling tired or having a lack of energy
- diarrhoea
- loss of appetite
- feeling sick or vomiting


# Step 10 — Validate the 5-condition preview
## Before touching all 406 records, check that:
### - every demo condition produced at least one phrase
### - there are no empty phrases
### - there are no exact duplicate phrases
### - obvious source/navigation text is not present

In [65]:
preview_results = []

NOISE_TERMS = [
    "macmillan cancer support",
    "read more about",
    "there's a separate page",
    "you can also get advice",
    "images on this page",
]

for record in demo_records:
    phrases = parse_symptoms(record["symptoms"])

    preview_results.append({
        "condition": record["condition"],
        "symptom_phrases": phrases,
        "phrase_count": len(phrases),
    })

for result in preview_results:
    phrases = result["symptom_phrases"]
    lowered = [p.casefold() for p in phrases]

    duplicate_count = len(lowered) - len(set(lowered))
    empty_count = sum(not p.strip() for p in phrases)
    noise_count = sum(
        any(term in p.casefold() for term in NOISE_TERMS)
        for p in phrases
    )

    print(result["condition"])
    print("  phrase_count:", result["phrase_count"])
    print("  duplicates:", duplicate_count)
    print("  empty_phrases:", empty_count)
    print("  obvious_noise:", noise_count)
    print()

    assert result["phrase_count"] > 0
    assert duplicate_count == 0
    assert empty_count == 0
    assert noise_count == 0

print("5-condition preview validation: PASSED")

Acute pancreatitis
  phrase_count: 7
  duplicates: 0
  empty_phrases: 0
  obvious_noise: 0

Acne
  phrase_count: 6
  duplicates: 0
  empty_phrases: 0
  obvious_noise: 0

Scoliosis
  phrase_count: 7
  duplicates: 0
  empty_phrases: 0
  obvious_noise: 0

Aplastic anaemia
  phrase_count: 9
  duplicates: 0
  empty_phrases: 0
  obvious_noise: 0

Acute respiratory infection (ARI)
  phrase_count: 13
  duplicates: 0
  empty_phrases: 0
  obvious_noise: 0

5-condition preview validation: PASSED


# Step 11 — Preview the structured output

In [66]:
for result in preview_results:
    print(json.dumps(
        result,
        ensure_ascii=False,
        indent=2,
    ))
    print("-" * 70)

{
  "condition": "Acute pancreatitis",
  "symptom_phrases": [
    "a severe, dull pain around the top of your stomach that develops suddenly",
    "nausea (feeling sick) or vomiting",
    "diarrhoea",
    "indigestion",
    "a high temperature (fever) of 38C (100.4F) or above",
    "jaundice - yellowing of the skin and the whites of the eyes",
    "tenderness or swelling of the abdomen (tummy)"
  ],
  "phrase_count": 7
}
----------------------------------------------------------------------
{
  "condition": "Acne",
  "symptom_phrases": [
    "blackheads - small black or yellowish bumps on the skin",
    "whiteheads - similar to blackheads but may be firmer",
    "papules - small red bumps that may feel tender or sore",
    "pustules - similar to papules, but have a white tip in the centre caused by a build-up of pus",
    "nodules - large hard lumps that build up beneath the surface of the skin and can be painful",
    "cysts - severe acne can cause nodules and cysts which are large pu

# Step 12 — Full 406-condition parser (disabled for now)
## Do not enable this until the 5 preview outputs have been reviewed.
## This is intentional. Different NHS pages use different writing styles, so it is safer to refine the extraction rules before generating the final dataset.

In [67]:
CHECK_CONDITIONS = [
    "Acute lymphoblastic leukaemia",
    "Acne",
    "Addison’s disease",
    "Cirrhosis",
    "Scoliosis",
    "Achilles tendinopathy",
    "Coronavirus (COVID-19)",
    "Erectile dysfunction (impotence)",
    "Bowel cancer",
    "Allergic rhinitis",
]

for name in CHECK_CONDITIONS:

    record = next(
        r for r in conditions
        if r["condition"] == name
    )

    phrases = parse_symptoms(
        record["symptoms"]
    )

    print()
    print("=" * 70)
    print(name)
    print("=" * 70)

    for phrase in phrases:
        print("-", phrase)



Acute lymphoblastic leukaemia
- swollen lymph nodes (lumps in your neck, armpits or groins), especially if they remain swollen for 6 weeks or more
- fatigue
- drenching night sweats
- rash or itchy skin that's unexplained
- paleness
- bruising or bleeding with no obvious cause
- losing weight without trying
- new shortness of breath
- infections that are persistent or keep coming back after treatment
- pain in your bones or abdomen (tummy)

Acne
- blackheads - small black or yellowish bumps on the skin
- whiteheads - similar to blackheads but may be firmer
- papules - small red bumps that may feel tender or sore
- pustules - similar to papules, but have a white tip in the centre caused by a build-up of pus
- nodules - large hard lumps that build up beneath the surface of the skin and can be painful
- cysts - severe acne can cause nodules and cysts which are large pus-filled lumps that can lead to scarring

Addison’s disease
- fatigue (lack of energy or motivation)
- lethargy (abnormal

In [68]:

QUALITY_NOISE_RE = re.compile(
    r"(?:"
    r"has more information|"
    r"provides information|"
    r"support,\s*advice and information|"
    r"macmillan cancer support|"
    r"managing .+ treatments?|"
    r"\bivf\b|"
    r"donated eggs|"
    r"treatment with a surrogate|"
    r"life expectancy|"
    r"how many cells contain the extra chromosome|"
    r"how much of the extra chromosome"
    r")",
    flags=re.IGNORECASE,
)


def needs_parser_review(
    condition: str,
    symptom_text: str,
    phrases: list[str],
    parser_method: str,
) -> bool:

    # Nothing could be extracted.
    if not phrases:
        return True

    # Narrative fallbacks remain intentionally review-only.
    if parser_method == "narrative_fallback":
        return True

    # A heading fragment must never become a model symptom.
    if any(
        is_incomplete_leadin_phrase(phrase)
        for phrase in phrases
    ):
        return True

    # Obvious source navigation / treatment / non-symptom noise.
    if any(
        QUALITY_NOISE_RE.search(phrase)
        for phrase in phrases
    ):
        return True

    # A huge capture probably means the parser grabbed prose.
    if any(
        len(phrase.split()) > 35
        for phrase in phrases
    ):
        return True

    # Lots of source text but almost nothing extracted.
    if (
        len(symptom_text.split()) > 80
        and len(phrases) < 2
    ):
        return True

    # Abnormally large extraction.
    if len(phrases) > 30:
        return True

    return False


In [69]:
RUN_FULL_DATASET = True

if not RUN_FULL_DATASET:
    print(
        "Full-dataset parsing is currently DISABLED.\n"
        "Review the demo conditions first.\n"
        "When ready, change RUN_FULL_DATASET = True."
    )

else:
    parsed_conditions = []

    for record in conditions:

        
        phrases, parser_method = parse_symptoms(
            record["symptoms"],
            return_method=True,
        )

        parsed_record = dict(record)

        parsed_record["symptom_phrases"] = phrases

        parsed_record["symptom_phrase_count"] = len(
            phrases
        )

        parsed_record["parser_method"] = parser_method

        parsed_record["parser_needs_review"] = (
            needs_parser_review(
                record["condition"],
                record["symptoms"],
                phrases,
                parser_method,
            )
        )

        parsed_conditions.append(
            parsed_record
        )

    # -----------------------------------
    # Validation / summary
    # -----------------------------------


    from collections import Counter

    method_counts = Counter(
        record["parser_method"]
        for record in parsed_conditions
    )

    print("\nParser methods")
    print("--------------")

    for method, count in method_counts.items():
        print(
            f"{method}: {count}"
        )

    review_records = [
        record
        for record in parsed_conditions
        if record["parser_needs_review"]
    ]

    zero_phrase_records = [
        record
        for record in parsed_conditions
        if record["symptom_phrase_count"] == 0
    ]

    print("Parser summary")
    print("--------------")

    print(
        "Total conditions:",
        len(parsed_conditions),
    )

    print(
        "Conditions with phrases:",
        sum(
            record["symptom_phrase_count"] > 0
            for record in parsed_conditions
        ),
    )

    print(
        "Zero phrase conditions:",
        len(zero_phrase_records),
    )

    print(
        "Needs review:",
        len(review_records),
    )

    print("\nFirst review cases:")

    for record in review_records[:20]:
        print(
            "-",
            record["condition"],
            "| phrases:",
            record["symptom_phrase_count"],
        )


Parser methods
--------------
structured: 301
none: 73
plain_list_fallback: 25
narrative_fallback: 7
Parser summary
--------------
Total conditions: 406
Conditions with phrases: 333
Zero phrase conditions: 73
Needs review: 88

First review cases:
- Acid and chemical burns | phrases: 0
- Angioedema | phrases: 1
- Arthritis | phrases: 0
- Bacterial vaginosis | phrases: 1
- Blisters | phrases: 0
- Bone cancer | phrases: 1
- Bow legs and knock knees in children and young people | phrases: 0
- Bowel incontinence | phrases: 0
- Brain stem death | phrases: 0
- Breast pain | phrases: 0
- Breast swelling in men | phrases: 0
- Cardiac arrest | phrases: 0
- Carpal tunnel syndrome | phrases: 2
- Chest pain | phrases: 2
- Chronic obstructive pulmonary disease (COPD) | phrases: 0
- Chronic pain | phrases: 0
- Chronic pancreatitis | phrases: 1
- Coma | phrases: 0
- Cough | phrases: 0
- Croup | phrases: 0


In [70]:
# ---------------------------------------------------------
# Narrative fallback audit
# ---------------------------------------------------------

narrative_records = [
    record
    for record in parsed_conditions
    if record["parser_method"] == "narrative_fallback"
]

none_records = [
    record
    for record in parsed_conditions
    if record["parser_method"] == "none"
]

print("Narrative fallback audit")
print("------------------------")
print("Narrative fallback:", len(narrative_records))
print("Still none:", len(none_records))

print("\n" + "=" * 80)
print("NARRATIVE FALLBACK RECORDS")
print("=" * 80)

for record in narrative_records:

    print("\nCONDITION:")
    print(record["condition"])

    print("\nSOURCE SYMPTOMS:")
    print(record["symptoms"])

    print("\nEXTRACTED:")
    for phrase in record["symptom_phrases"]:
        print("-", phrase)

    print("\n" + "-" * 80)


print("\n" + "=" * 80)
print("ZERO-PHRASE RECORDS")
print("=" * 80)

for record in none_records:

    print(
        "-",
        record["condition"]
    )

Narrative fallback audit
------------------------
Narrative fallback: 7
Still none: 73

NARRATIVE FALLBACK RECORDS

CONDITION:
Bacterial vaginosis

SOURCE SYMPTOMS:
Often there are no symptoms of bacterial vaginosis. Some women may notice a change in the normal discharge from the vagina. This discharge will usually be white or grey, thin or watery and have a strong, unpleasant fishy smell. This can be more noticeable during and after sex, and during periods. Bacterial vaginosis does not usually cause itching or irritation.

EXTRACTED:
- a change in the normal discharge from the vagina

--------------------------------------------------------------------------------

CONDITION:
Bone cancer

SOURCE SYMPTOMS:
A tumour in a bone can cause symptoms which include bone pain or swelling in the affected area that doesn't go away. Pain and swelling can be caused by other conditions, but it's important to speak to your doctor if you're concerned. Bone cancer is sometimes found when a bone breaks 

In [71]:

# ---------------------------------------------------------
# Full-dataset parser quality gate
# ---------------------------------------------------------

colon_header_leaks = []
incomplete_leadins = []
no_symptom_phrases = []
quality_noise = []

for record in parsed_conditions:

    source_colon_headers = {
        normalize_phrase(line)
        for line in record["symptoms"].splitlines()
        if line.strip().endswith(":")
    }

    for phrase in record["symptom_phrases"]:

        if phrase in source_colon_headers:
            colon_header_leaks.append(
                (record["condition"], phrase)
            )

        if is_incomplete_leadin_phrase(phrase):
            incomplete_leadins.append(
                (record["condition"], phrase)
            )

        if re.search(
            r"\b(?:no|without)\s+(?:noticeable\s+)?symptoms?\b",
            phrase,
            flags=re.IGNORECASE,
        ):
            no_symptom_phrases.append(
                (record["condition"], phrase)
            )

        if QUALITY_NOISE_RE.search(phrase):
            quality_noise.append(
                (record["condition"], phrase)
            )


print("Parser quality gate")
print("-------------------")
print("Colon-heading leaks:", len(colon_header_leaks))
print("Incomplete lead-ins:", len(incomplete_leadins))
print("'No symptoms' phrases:", len(no_symptom_phrases))
print("Quality-noise phrases:", len(quality_noise))

assert not colon_header_leaks, (
    "A colon-introduced list heading was stored as a symptom phrase."
)

assert not incomplete_leadins, (
    "An incomplete list lead-in was stored as a symptom phrase."
)

assert not no_symptom_phrases, (
    "A phrase describing absence of symptoms was stored as a symptom."
)

print("\nCore parser quality checks: PASSED ✅")

if quality_noise:
    print(
        "\nQuality-noise phrases remain review-only via "
        "parser_needs_review:"
    )

    for condition, phrase in quality_noise[:20]:
        print("-", condition, "->", phrase)


Parser quality gate
-------------------
Colon-heading leaks: 0
Incomplete lead-ins: 0
'No symptoms' phrases: 0
Quality-noise phrases: 3

Core parser quality checks: PASSED ✅

Quality-noise phrases remain review-only via parser_needs_review:
- Infertility -> IVF/ICSI
- Infertility -> more rarely, treatment with a surrogate
- Mesothelioma -> Macmillan Cancer Support has more information on mesothelioma symptoms


In [72]:
review_records = [
    record
    for record in parsed_conditions
    if record["parser_needs_review"]
]

zero_phrase_records = [
    record
    for record in parsed_conditions
    if record["symptom_phrase_count"] == 0
]

print("Parser summary")
print("--------------")
print("Total conditions:", len(parsed_conditions))
print(
    "Conditions with phrases:",
    sum(
        record["symptom_phrase_count"] > 0
        for record in parsed_conditions
    ),
)
print(
    "Zero phrase conditions:",
    len(zero_phrase_records),
)
print(
    "Needs review:",
    len(review_records),
)

print("\nFirst review cases:")

for record in review_records[:20]:
    print(
        "-",
        record["condition"],
        "| phrases:",
        record["symptom_phrase_count"],
    )

Parser summary
--------------
Total conditions: 406
Conditions with phrases: 333
Zero phrase conditions: 73
Needs review: 88

First review cases:
- Acid and chemical burns | phrases: 0
- Angioedema | phrases: 1
- Arthritis | phrases: 0
- Bacterial vaginosis | phrases: 1
- Blisters | phrases: 0
- Bone cancer | phrases: 1
- Bow legs and knock knees in children and young people | phrases: 0
- Bowel incontinence | phrases: 0
- Brain stem death | phrases: 0
- Breast pain | phrases: 0
- Breast swelling in men | phrases: 0
- Cardiac arrest | phrases: 0
- Carpal tunnel syndrome | phrases: 2
- Chest pain | phrases: 2
- Chronic obstructive pulmonary disease (COPD) | phrases: 0
- Chronic pain | phrases: 0
- Chronic pancreatitis | phrases: 1
- Coma | phrases: 0
- Cough | phrases: 0
- Croup | phrases: 0


# Step 17 — Save final parser output (only when full run is enabled)
## When enabled, the parsed dataset will be saved to:
### app/dataset/processed/conditions_symptom_parsed.json
## The original MongoDB data and conditions_text_cleaned.json remain untouched.

In [73]:
from pathlib import Path
import json

if RUN_FULL_DATASET:

    OUTPUT_DIR = PROJECT_ROOT / "app" / "dataset" / "output" / "preprocessing"

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    OUTPUT_PATH = OUTPUT_DIR / "conditions_symptom_parsed.json"

    with OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(
            parsed_conditions,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("Saved:")
    print(OUTPUT_PATH)

else:
    print("Nothing saved yet — preview mode only.")

Saved:
C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_symptom_parsed.json
